In [ ]:
# Imports
import sys
sys.path.append("Program")

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_infix
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
import pandas as pd
from plot import *
from scipy.optimize import minimize
from technicals import *
from tqdm import tqdm

# Start of the program
start = dt.datetime.now()

# Variables
all_stocks = False
period_short = 60
period_long = 252
RS = 90
factors = [0.1, 0.55, 0.35]
backtest = False

# Index
index_name = "^HSI"
index_dict = {"^HSI": "HKEX", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Get the infix
infix = get_infix("^GSPC", index_dict, all_stocks) 

# Modify the current date
current_date = modify_current_date(start, index_name)

In [ ]:
stocks = ["3988.HK", "3328.HK", "0939.HK", "1398.HK", 
          "2638.HK", "1810.HK", "0857.HK", "0386.HK", "0883.HK", "0062.HK", 
          "1503.HK", "1881.HK", "1426.HK", "0808.HK", "0435.HK", "0778.HK", "0405.HK", "2778.HK"]
stocks_num = len(stocks)

In [ ]:
# Retrieve historical price data for each stock using get_df
price_data = pd.DataFrame()
for stock in tqdm(stocks):
    df = get_df(stock, current_date, adj=True)
    price_data[stock] = df["Close"]

# Compute daily returns
returns = price_data.pct_change(fill_method=None).dropna()

# Calculate the mean returns and the covariance matrix of returns
mean_returns = returns.mean()
cov_matrix = returns.cov()

# Set the number of portfolios to simulate
num_portfolios = 10000

# Initialize arrays to store simulation results
results = np.zeros((3, num_portfolios))
weights_record = []

for i in range(num_portfolios):
    # Generate random portfolio weights
    weights = np.random.random(len(stocks))
    weights /= np.sum(weights)
    weights_record.append(weights)
    
    # Calculate annualized portfolio return and volatility
    portfolio_return = np.sum(mean_returns * weights) * 252
    portfolio_stddev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix * 252, weights)))
    
    # Calculate Sharpe ratio (assuming risk-free rate = 0)
    sharpe_ratio = portfolio_return / portfolio_stddev
    
    results[0, i] = portfolio_return
    results[1, i] = portfolio_stddev
    results[2, i] = sharpe_ratio

# Identify the portfolio with the highest Sharpe ratio
max_sharpe_idx = np.argmax(results[2])
optimal_weights = weights_record[max_sharpe_idx]

# Display the optimal portfolio weights
optimal_portfolio = pd.Series(optimal_weights, index=stocks)
print("Optimal Weights:\n", optimal_portfolio)

# Compute and plot the equity curve for the optimal portfolio
equity_curve = (returns @ optimal_weights + 1).cumprod()
plt.figure(figsize=(10, 6))
plt.plot(equity_curve, label="Optimal Portfolio")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.title("Equity Curve of Optimal Portfolio")
plt.show()

# Plot the efficient frontier
plt.figure(figsize=(10, 6))
plt.scatter(results[1, :], results[0, :], c=results[2, :], cmap="viridis", marker="o", s=10, alpha=0.7)
plt.xlabel("Volatility")
plt.ylabel('Return')
plt.title("Efficient Frontier")
plt.colorbar(label="Sharpe Ratio")
plt.scatter(results[1, max_sharpe_idx], results[0, max_sharpe_idx], c="red", s=50, label="Max Sharpe Ratio")
plt.legend()
plt.show()